In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy
import PIL
import requests

In [3]:
# load data to df
df = pd.read_csv(
    "C:\\Users\\papak\\_Study\\DSI\\visualization\\02_activities\\assignments\\Live Births by Age of Parents.csv"
)

# display first 10 rows of df
df.head(10)

,_id,Year/Année,Parent who gave birth/Parent qui a donné naissance,Age of Father or Parent / Âge du père ou parent,Births/Naissances
0,1,2012,15,<20,59
1,2,2012,15,20-24,7
2,3,2012,15,25-29,1
3,4,2012,15,30-34,0
4,5,2012,15,35-39,0
5,6,2012,15,40-44,0
6,7,2012,15,45-49,0
7,8,2012,15,50-54,0
8,9,2012,15,55-59,0
9,10,2012,15,60+,0


In [4]:
# display column names of df
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4023 entries, 0 to 4022
Data columns (total 5 columns):
 #   Column                                              Non-Null Count  Dtype 
---  ------                                              --------------  ----- 
 0   _id                                                 4023 non-null   int64 
 1   Year/Année                                          4023 non-null   int64 
 2   Parent who gave birth/Parent qui a donné naissance  4021 non-null   object
 3   Age of Father or Parent / Âge du père ou parent     4023 non-null   object
 4   Births/Naissances                                   4023 non-null   int64 
dtypes: int64(3), object(2)
memory usage: 157.3+ KB


In [21]:
year = "Year/Année"
mother_age = "Parent who gave birth/Parent qui a donné naissance"
father_age = "Age of Father or Parent / Âge du père ou parent"
birth_count = "Births/Naissances"

In [22]:
# check unique vaules in 'Age of Mother' column
df[mother_age].unique()

array(['15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25',
       '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36',
       '37', '38', '39', '40', '41', '42', '43', '44', '<15', '>44',
       'N.S./N.P.', 'NS/N.P.', nan], dtype=object)

In [28]:
# show rows contains 'N.S./N.P.', 'NS/N.P.', nan in 'Age of Mother' column
df[df[mother_age].isin(["N.S./N.P.", "NS/N.P."])]


,_id,Year/Année,Parent who gave birth/Parent qui a donné naissance,Age of Father or Parent / Âge du père ou parent,Births/Naissances
352,353,2012,N.S./N.P.,<20,0
353,354,2012,N.S./N.P.,20-24,1
354,355,2012,N.S./N.P.,25-29,0
355,356,2012,N.S./N.P.,30-34,0
356,357,2012,N.S./N.P.,35-39,3
...,...,...,...,...,...
2553,2554,2019,NS/N.P.,NS/N.P.,89
2858,2859,2020,NS/N.P.,20-24,2
2859,2860,2020,NS/N.P.,NS/N.P.,87
3156,3157,2021,NS/N.P.,20-24,1


In [29]:
df[df[mother_age].isnull()]

,_id,Year/Année,Parent who gave birth/Parent qui a donné naissance,Age of Father or Parent / Âge du père ou parent,Births/Naissances
3738,3739,2023,NaN,25-29,1
3739,3740,2023,NaN,NS/N.P.,95


In [25]:
# check unique value in 'Age of Father' column
df[father_age].unique()

array(['<20', '20-24', '25-29', '30-34', '35-39', '40-44', '45-49',
       '50-54', '55-59', '60+', 'N.S./N.P.', 'NS/N.P.'], dtype=object)

In [31]:
# show rows contains 'N.S./N.P.', 'NS/N.P.', nan in 'Age of Father' column
df[df[father_age].isin(["N.S./N.P.", "NS/N.P."])]

,_id,Year/Année,Parent who gave birth/Parent qui a donné naissance,Age of Father or Parent / Âge du père ou parent,Births/Naissances
10,11,2012,15,N.S./N.P.,33
21,22,2012,16,N.S./N.P.,104
32,33,2012,17,N.S./N.P.,219
43,44,2012,18,N.S./N.P.,273
54,55,2012,19,N.S./N.P.,357
...,...,...,...,...,...
3983,3984,2023,40,NS/N.P.,135
3993,3994,2023,41,NS/N.P.,105
4003,4004,2023,42,NS/N.P.,71
4012,4013,2023,43,NS/N.P.,38


In [26]:
# check unique value in Year column
df[year].unique()

array([2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022,
       2023])

In [27]:
# check if Birth count column has null value or non-numeric value
print(df[birth_count].isnull().sum())
df[birth_count].dtype

0


dtype('int64')

Want to clean Age of Mother and Father data

In [35]:
def clean_age(value): 
    if pd.isna(value): return "Unknown" # keep NaN as "Unknown"
    elif value == "N.S./N.P.": return "Unknown" # keep as a category 
    elif value == "NS/N.P.": return "Unknown" # keep as a category
    elif str(value).startswith("<"): 
        upper = int(str(value)[1:]) - 1 # e.g. "<20" → 19
        return f"0-{upper}" # e.g. "<20" → "0-19" 
    elif str(value).startswith(">"): 
        lower = int(str(value)[1:]) + 1 # e.g. ">44" → 45
        return f"{lower}+" # e.g. ">44" → "45+" 
    elif "-" in str(value): return str(value) # keep ranges as-is 
    elif str(value).isdigit(): return str(value) # single age 
    else: return "Unknown" 

df["Mother_Age_group"] = df[mother_age].apply(clean_age) 
df["Father_Age_group"] = df[father_age].apply(clean_age)
print(df[["Father_Age_group","Mother_Age_group"]].head())

  Father_Age_group Mother_Age_group
0             0-19               15
1            20-24               15
2            25-29               15
3            30-34               15
4            35-39               15


In [36]:
#double check
print(clean_age("<20"))   # "0-19"
print(clean_age(">44"))   # "45+"
print(clean_age("20-24")) # "20-24"
print(clean_age("N.S./N.P.")) # "Unknown"


0-19
45+
20-24
Unknown


In [41]:
# Check unique values in cleaned columns
print("Mother Age groups:", df["Mother_Age_group"].unique())
print("Father Age groups:", df["Father_Age_group"].unique())


Mother Age groups: ['15' '16' '17' '18' '19' '20' '21' '22' '23' '24' '25' '26' '27' '28'
 '29' '30' '31' '32' '33' '34' '35' '36' '37' '38' '39' '40' '41' '42'
 '43' '44' '0-14' '45+' 'Unknown']
Father Age groups: ['0-19' '20-24' '25-29' '30-34' '35-39' '40-44' '45-49' '50-54' '55-59'
 'Unknown']


Start making graph - Study Underage Pregancy

In [63]:
# filter underage mother (<=17 years old)
underage = df[df["Mother_Age_group"].isin(["0-14", "16", "17"])]

# aggregate birth count by year
underage_trend = underage.groupby(year)[birth_count].sum().reset_index()
print(underage_trend)

    Year/Année  Births/Naissances
0         2012               1044
1         2013                897
2         2014                729
3         2015                724
4         2016                548
5         2017                551
6         2018                423
7         2019                401
8         2020                355
9         2021                282
10        2022                291
11        2023                257


In [64]:
father_combo = underage.groupby("Father_Age_group")[birth_count].sum().reset_index()
#sort by number of births (descending)
father_combo = father_combo.sort_values(by=birth_count, ascending=False)  
print(father_combo.head(10))

  Father_Age_group  Births/Naissances
0             0-19               2934
9          Unknown               1844
1            20-24               1458
2            25-29                199
3            30-34                 38
4            35-39                 13
5            40-44                 10
6            45-49                  4
7            50-54                  1
8            55-59                  1


In [ ]:
#Interactive bar chart of father age groups for underage mothers using plotly
import plotly.express as px
fig = px.bar(father_combo, 
                x="Father_Age_group", 
                y=birth_count, 
                title="Birth Counts by Father Age Groups of Underage Mothers (<=17 years old)",
                labels={"Father_Age_group": "Father Age Group", birth_count: "Birth Count"},
                hover_data=[birth_count] # show birth count on hover
                #, orientation='h'  # horizontal bars
                ) 
fig.update_layout(title={'x':0.5, 'xanchor':'center', # center and style title
                         'font': dict(size = 24, family = 'Impact, Arial Black, sans-serif', color = 'darkblue')}, #want it be a little 3D-like
                  xaxis_tickangle=-45, # rotate x-axis labels for better readability
                  width = 1200, #increawe figure width
                  height = 1000, #increase figure height
                  margin=dict(l=100, r=50, t=60, b=60), # l=eft, r=right, t=top, b=bottom margins
                  #add gridline:
                  xaxis=dict(showgrid=True, gridcolor='orange', gridwidth=0.5, griddash='dot', # add vertical gridlines with light gray color and dotted style
                             title_font = dict(size=20, family='Arial Bold', color = 'black') #bold the x-axis title
                             ), 
                  yaxis=dict(showgrid=True, gridcolor='orange', gridwidth=0.5, griddash='dash',
                             title_font = dict(size=20, family='Arial Bold', color = 'black') #bold the y-axis title
                             ),
                  plot_bgcolor='white', 
                  paper_bgcolor='lightgrey'
                  ) 
#fig.update_traces( hovertemplate="Births: %{y}<extra></extra>" ) # customize hover text to only show birth count for cleaner display
fig.update_traces( hovertemplate="Births: %{y}<extra></extra>",
                  marker = dict(line = dict(width = 1, color = "black")),
                   width = 0.8) # customize hover text and add borders to bars for better visibility
fig.show()

In [123]:
# output the interactive plot to html file
fig.write_html("assignment_3_visualization.html")